In [1]:
# CELL 1 — Imports & config

import requests
import gzip
import json
import os
from io import BytesIO
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

# Storage for ALL CRITICAL and HIGH CVEs (no limits - collect everything)
FILES = {
    #"CRITICAL": f"{DATA_DIR}/cves_all_critical.jsonl",
    #"HIGH": f"{DATA_DIR}/cves_all_high.jsonl",
    "MEDIUM": f"{DATA_DIR}/cves_all_medium.jsonl",
    "LOW": f"{DATA_DIR}/cves_all_low.jsonl",
}
PROGRESS_FILE = f"{DATA_DIR}/progress_json_all.json"

# NVD JSON 2.0 feed URL template
NVD_URL_TEMPLATE = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"
YEARS = list(range(2020, 2027))  # 2020-2026

print(f"Data directory: {DATA_DIR}")
print(f"Years to process: {YEARS}")
print(f"Storage files (collecting ALL MEDIUM + LOW, no limits):") 
for sev, path in FILES.items():
    print(f"  {sev}: {path}")


Data directory: ../data
Years to process: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
Storage files (collecting ALL MEDIUM + LOW, no limits):
  MEDIUM: ../data/cves_all_medium.jsonl
  LOW: ../data/cves_all_low.jsonl


In [2]:
# CELL 2 — Cleaning function (same as API pipeline)

def clean_cve(raw):
    """
    Extract and validate fields from a raw NVD CVE object.
    Returns a clean dict or None if the CVE fails quality checks.
    """
    cve = raw["cve"]
    metrics = cve.get("metrics", {})

    # Must have CVSS v3.1
    if "cvssMetricV31" not in metrics:
        return None

    cvss = metrics["cvssMetricV31"][0]["cvssData"]

    # Must have a description in English
    descriptions = [d for d in cve.get("descriptions", []) if d["lang"] == "en"]
    if not descriptions or len(descriptions[0]["value"]) < 100:
        return None

    # Must have CPE (affected product info)
    configurations = cve.get("configurations", [])
    if not configurations:
        return None

    return {
        "id": cve["id"],
        "published": cve["published"],
        "lastModified": cve["lastModified"],
        "description": descriptions[0]["value"],
        "cvss_score": cvss["baseScore"],
        "cvss_severity": cvss["baseSeverity"],
        "attack_vector": cvss["attackVector"],
        "attack_complexity": cvss["attackComplexity"],
        "privileges_required": cvss["privilegesRequired"],
        "user_interaction": cvss["userInteraction"],
        "confidentiality_impact": cvss["confidentialityImpact"],
        "integrity_impact": cvss["integrityImpact"],
        "availability_impact": cvss["availabilityImpact"],
        "configurations": configurations,
        "references": [r["url"] for r in cve.get("references", [])]
    }

In [3]:
# CELL 3 — Helper functions

def load_progress():
    """Load progress tracking."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"years_processed": [], "counts": {"MEDIUM": 0, "LOW": 0}}

def save_progress(progress):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def load_existing_ids():
    """Load all already-stored CVE IDs to avoid duplicates."""
    seen_ids = set()
    for severity in ["MEDIUM", "LOW"]:
        path = FILES[severity]
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        seen_ids.add(json.loads(line)["id"])
    return seen_ids

def get_current_counts():
    """Get current counts per severity."""
    counts = {"MEDIUM": 0, "LOW": 0}
    for severity in ["MEDIUM", "LOW"]:
        path = FILES[severity]
        if os.path.exists(path):
            with open(path) as f:
                counts[severity] = sum(1 for _ in f if _.strip())
    return counts

def download_and_parse_year(year):
    """Download NVD JSON for a year, gunzip, and return parsed data."""
    url = NVD_URL_TEMPLATE.format(year=year)
    print(f"\nDownloading {year}...", end=" ")
    
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"❌ HTTP {r.status_code}")
            return None
        
        print(f"({len(r.content) / 1024 / 1024:.1f} MB) → Decompressing...", end=" ")
        data = json.loads(gzip.decompress(r.content))
        cves = data.get("vulnerabilities", [])
        print(f"✅ {len(cves)} CVEs")
        return cves
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

In [4]:
# CELL 4 — Current status

seen_ids = load_existing_ids()
counts = get_current_counts()
progress = load_progress()

print(f"\n--- Current status ---")
print(f"Years processed so far: {progress['years_processed']}")
print(f"Stored CVEs (no limits, collecting all):") 
print(f"  MEDIUM: {counts['MEDIUM']}")
print(f"  LOW: {counts['LOW']}")
print(f"  Total: {counts['MEDIUM'] + counts['LOW']}")
print(f"Unique CVE IDs in storage: {len(seen_ids)}")



--- Current status ---
Years processed so far: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
Stored CVEs (no limits, collecting all):
  MEDIUM: 0
  LOW: 0
  Total: 0
Unique CVE IDs in storage: 0


In [5]:
# CELL 5 — Process JSON files and store ALL CRITICAL + HIGH

def run_json_pipeline():
    progress = load_progress()
    seen_ids = load_existing_ids()
    counts = get_current_counts()
    
    print(f"\n{'='*60}")
    print(f"Storing ALL NVD MEDIUM + LOW CVEs (2020-2026)")
    print(f"No limits - collecting everything that passes quality filter")
    print(f"{'='*60}")
    
    for year in YEARS:
        if year in progress["years_processed"]:
            print(f"\n[{year}] Already processed, skipping")
            continue
        
        cves = download_and_parse_year(year)
        if not cves:
            continue
        
        year_counts = {"MEDIUM": 0, "LOW": 0}
        
        with tqdm(total=len(cves), desc=f"{year}", unit="CVE") as pbar:
            for raw in cves:
                cve_id = raw["cve"]["id"]
                if cve_id in seen_ids:
                    pbar.update(1)
                    continue
                
                cleaned = clean_cve(raw)
                if cleaned:
                    severity = cleaned["cvss_severity"]
                    if severity in ["MEDIUM", "LOW"]:
                        # Store ALL MEDIUM and LOW (no limits)
                        with open(FILES[severity], "a") as f:
                            f.write(json.dumps(cleaned) + "\n")
                        seen_ids.add(cve_id)
                        counts[severity] += 1
                        year_counts[severity] += 1
                
                pbar.update(1)
        
        progress["years_processed"].append(year)
        save_progress(progress)
        
        print(f"   → Stored {year_counts['MEDIUM']} MEDIUM, {year_counts['LOW']} LOW")
        print(f"   → Running totals: MEDIUM {counts['MEDIUM']}, LOW {counts['LOW']} (Total: {counts['MEDIUM'] + counts['LOW']})")
    
    print(f"\n{'='*60}")
    print(f"✅ Complete! All years processed.")
    print(f"   CRITICAL: {counts['MEDIUM']} CVEs")
    print(f"   HIGH: {counts['LOW']} CVEs")
    print(f"   Total: {counts['MEDIUM'] + counts['LOW']} CVEs")
    print(f"   Files ready for filtering strategy:")
    for sev, path in FILES.items():
        print(f"     {sev}: {path}")


In [7]:
# CELL 6 — Run the pipeline

run_json_pipeline()


Storing ALL NVD MEDIUM + LOW CVEs (2020-2026)
No limits - collecting everything that passes quality filter



2020: 100%|███████████████████████████████████████████| 21050/21050 [00:00<00:00, 39778.45CVE/s]


   → Stored 7479 MEDIUM, 502 LOW
   → Running totals: MEDIUM 7479, LOW 502 (Total: 7981)



2021: 100%|███████████████████████████████████████████| 23428/23428 [00:00<00:00, 46258.72CVE/s]


   → Stored 8932 MEDIUM, 720 LOW
   → Running totals: MEDIUM 16411, LOW 1222 (Total: 17633)



...

   → Stored 15022 MEDIUM, 1358 LOW
   → Running totals: MEDIUM 55939, LOW 4551 (Total: 60490)



2025: 100%|███████████████████████████████████████████| 44651/44651 [00:00<00:00, 70593.57CVE/s]


   → Stored 11753 MEDIUM, 1494 LOW
   → Running totals: MEDIUM 67692, LOW 6045 (Total: 73737)



2026: 100%|███████████████████████████████████████████| 26104/26104 [00:00<00:00, 74179.74CVE/s]


   → Stored 6712 MEDIUM, 609 LOW
   → Running totals: MEDIUM 74404, LOW 6654 (Total: 81058)

✅ Complete! All years processed.
   CRITICAL: 74404 CVEs
   HIGH: 6654 CVEs
   Total: 81058 CVEs
   Files ready for filtering strategy:
     MEDIUM: ../data/cves_all_medium.jsonl
     LOW: ../data/cves_all_low.jsonl


SyntaxError: invalid syntax (3012382316.py, line 1)